# StockLens — LSTM Training & ONNX Export
Entrena un LSTM real con PyTorch (BPTT completo) y exporta a ONNX para Railway.

**Instrucciones:**
1. Ejecuta todas las celdas en orden
2. En la celda de configuración, cambia `TICKERS` por las acciones que quieras
3. Al final, descarga los archivos `.onnx` y `.pkl` y súbelos a `stocklens-python/app/models/`

In [ ]:
# ── Celda 1: Instalación de dependencias ──────────────────────────────────────
!pip install -q yfinance pandas-ta shap onnx onnxruntime torch torchvision
print('✅ Dependencias instaladas')

In [ ]:
# ── Celda 2: Configuración ────────────────────────────────────────────────────
# 🔧 EDITA AQUÍ: lista de tickers a entrenar
TICKERS = ['AAPL', 'NVDA', 'MSFT', 'JPM', 'BIIB', 'UBER', 'TSLA']

TIMESTEPS   = 30      # ventana de historia para cada predicción
HIDDEN_SIZE = 64      # neuronas en la capa LSTM
NUM_LAYERS  = 2       # capas LSTM apiladas
DROPOUT     = 0.25    # dropout entre capas (para MC Dropout en inferencia)
EPOCHS      = 80      # épocas de entrenamiento
LR          = 0.001   # learning rate
BATCH_SIZE  = 32
TRAIN_SPLIT = 0.85    # 85% train, 15% test
PERIOD      = '3y'    # historial de datos
MC_SAMPLES  = 50      # muestras Monte Carlo para incertidumbre

import os
os.makedirs('models', exist_ok=True)
print(f'✅ Configuración lista — {len(TICKERS)} tickers a entrenar')

In [ ]:
# ── Celda 3: Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_ta as ta
import pickle, json, shutil
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error
import torch
import torch.nn as nn
import torch.onnx
import shap
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Usando device: {device}')

In [ ]:
# ── Celda 4: Feature engineering ─────────────────────────────────────────────
def build_features(ticker: str, period: str = PERIOD) -> tuple[pd.DataFrame, list[str]]:
    df = yf.Ticker(ticker).history(period=period, interval='1d', auto_adjust=True)
    if df is None or len(df) < 100:
        raise ValueError(f'Datos insuficientes para {ticker}')

    closes = df['Close']; highs = df['High']; lows = df['Low']; vols = df['Volume']

    df.ta.sma(length=10, close=closes, append=True)
    df.ta.sma(length=20, close=closes, append=True)
    df.ta.sma(length=50, close=closes, append=True)
    df.ta.ema(length=12, close=closes, append=True)
    df.ta.ema(length=26, close=closes, append=True)
    df.ta.rsi(length=14, close=closes, append=True)
    df.ta.macd(fast=12, slow=26, signal=9, close=closes, append=True)
    df.ta.bbands(length=20, std=2, close=closes, append=True)
    df.ta.atr(length=14, high=highs, low=lows, close=closes, append=True)
    df.ta.stoch(high=highs, low=lows, close=closes, append=True)
    df.ta.obv(close=closes, volume=vols, append=True)

    for n in [1, 2, 3, 5, 10, 20]:
        df[f'ret_{n}d'] = closes.pct_change(n)

    df['vol_ratio']   = vols / (vols.rolling(20).mean() + 1e-9)
    df['high_low_pct']= (highs - lows) / (closes + 1e-9)
    df['close_open']  = (closes - df['Open']) / (df['Open'] + 1e-9)

    # Distance from moving averages
    for ma in ['SMA_10','SMA_20','SMA_50']:
        if ma in df.columns:
            df[f'dist_{ma.lower()}'] = (closes - df[ma]) / (df[ma] + 1e-9)

    # Rolling volatility (normalized target uses this)
    df['vol_20d'] = closes.pct_change().rolling(20).std()

    # Select feature columns
    exclude = {'Open','High','Low','Close','Volume','Dividends','Stock Splits'}
    feat_cols = [c for c in df.columns
                 if c not in exclude
                 and not c.startswith('BBB')
                 and df[c].dtype in [np.float64, np.float32, float]]

    df = df[['Close'] + feat_cols].dropna()
    return df, feat_cols

print('✅ Feature engineering definido')

In [ ]:
# ── Celda 5: Modelo LSTM PyTorch ──────────────────────────────────────────────
class StockLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = HIDDEN_SIZE,
                 num_layers: int = NUM_LAYERS, dropout: float = DROPOUT):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)  # for MC Dropout at inference
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, x):  # x: (batch, timesteps, features)
        out, _ = self.lstm(x)
        last    = self.dropout(out[:, -1, :])  # last timestep
        return self.fc(last).squeeze(-1)

print('✅ Arquitectura LSTM definida')

In [ ]:
# ── Celda 6: Pipeline de entrenamiento ───────────────────────────────────────
def build_sequences(X_scaled, y, timesteps=TIMESTEPS):
    Xs, ys = [], []
    for i in range(timesteps, len(X_scaled)):
        Xs.append(X_scaled[i-timesteps:i])
        ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


def train_ticker(ticker: str) -> dict:
    print(f'\n{'='*50}')
    print(f'Entrenando {ticker}...')

    # 1. Features
    df, feat_cols = build_features(ticker)
    closes   = df['Close'].values
    X_raw    = df[feat_cols].values
    n        = len(X_raw)

    # 2. Target: retorno a 1 día (para autoregresión iterativa)
    y_raw = np.zeros(n)
    y_raw[:-1] = (closes[1:] - closes[:-1]) / (closes[:-1] + 1e-9)
    y_raw[-1]  = 0.0  # último día sin target

    # 3. Scale
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    X_scaled = scaler_X.fit_transform(X_raw)
    y_scaled = scaler_y.fit_transform(y_raw.reshape(-1,1)).ravel()

    # 4. Sequences
    X_seq, y_seq = build_sequences(X_scaled, y_scaled)
    split  = int(len(X_seq) * TRAIN_SPLIT)
    X_tr, y_tr = X_seq[:split], y_seq[:split]
    X_te, y_te = X_seq[split:], y_seq[split:]

    # 5. DataLoaders
    tr_ds = torch.utils.data.TensorDataset(
        torch.tensor(X_tr), torch.tensor(y_tr))
    tr_dl = torch.utils.data.DataLoader(
        tr_ds, batch_size=BATCH_SIZE, shuffle=True)

    # 6. Model + optimizer
    model = StockLSTM(input_size=X_raw.shape[1]).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)
    loss_fn = nn.HuberLoss()  # más robusto que MSE para outliers financieros

    # 7. Training loop con early stopping
    best_val_loss = float('inf')
    best_state    = None
    patience_cnt  = 0
    PATIENCE      = 15

    X_te_t = torch.tensor(X_te).to(device)
    y_te_t = torch.tensor(y_te).to(device)

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for xb, yb in tr_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()  # ✅ BPTT real — PyTorch propaga por todos los timesteps
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # evita exploding gradients
            opt.step()
            epoch_loss += loss.item()

        # Validation
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_te_t), y_te_t).item()
        sched.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt  = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f'  Early stop at epoch {epoch+1}')
                break

        if (epoch+1) % 20 == 0:
            print(f'  Epoch {epoch+1:3d} — train: {epoch_loss/len(tr_dl):.5f} | val: {val_loss:.5f}')

    # 8. Load best weights
    model.load_state_dict(best_state)
    model.eval()

    # 9. Test MAPE (en espacio original, no normalizado)
    with torch.no_grad():
        y_pred_norm = model(X_te_t).cpu().numpy()
    y_pred_ret = scaler_y.inverse_transform(y_pred_norm.reshape(-1,1)).ravel()
    y_true_ret = scaler_y.inverse_transform(y_te.reshape(-1,1)).ravel()

    # MAPE sobre retornos (evitar división por cero con retornos ~0)
    mask = np.abs(y_true_ret) > 0.001
    mape = float(np.mean(np.abs((y_true_ret[mask] - y_pred_ret[mask]) / y_true_ret[mask])) * 100) if mask.any() else 0.0
    print(f'  ✅ MAPE sobre retornos: {mape:.2f}%')
    print(f'  ✅ Samples: {len(X_tr)} train / {len(X_te)} test')

    # 10. SHAP feature importance (real, no uniforme)
    print('  Calculando SHAP...')
    model.train()  # activar dropout para SHAP
    background = torch.tensor(X_tr[:50]).to(device)
    explainer  = shap.GradientExplainer(model, background)
    shap_vals  = explainer.shap_values(torch.tensor(X_te[:30]).to(device))
    # shap_vals shape: (samples, timesteps, features) → mean over samples and timesteps
    importance_raw = np.abs(shap_vals).mean(axis=(0,1))  # (features,)
    importance_norm = importance_raw / (importance_raw.sum() + 1e-9)
    feat_importance = dict(sorted(
        zip(feat_cols, [round(float(v), 4) for v in importance_norm]),
        key=lambda x: x[1], reverse=True
    )[:10])
    print(f'  Top features: {list(feat_importance.items())[:3]}')
    model.eval()

    # 11. Export to ONNX
    dummy  = torch.zeros(1, TIMESTEPS, X_raw.shape[1]).to(device)
    onnx_path = f'models/lstm_{ticker}.onnx'
    torch.onnx.export(
        model, dummy, onnx_path,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
        opset_version=17,
        do_constant_folding=True,
    )
    print(f'  ✅ ONNX exportado → {onnx_path}')

    # 12. Save scalers + metadata
    meta = {
        'ticker':        ticker,
        'feat_cols':     feat_cols,
        'input_size':    X_raw.shape[1],
        'hidden_size':   HIDDEN_SIZE,
        'num_layers':    NUM_LAYERS,
        'dropout':       DROPOUT,
        'timesteps':     TIMESTEPS,
        'last_close':    float(closes[-1]),
        'last_X_scaled': X_scaled[-TIMESTEPS:].tolist(),  # last window for inference
        'feat_importance': feat_importance,
        'mape':          round(mape, 2),
        'train_samples': len(X_tr),
        'test_samples':  len(X_te),
        'period':        PERIOD,
    }
    pkl_path = f'models/lstm_{ticker}.pkl'
    with open(pkl_path, 'wb') as f:
        pickle.dump({'scaler_X': scaler_X, 'scaler_y': scaler_y, 'meta': meta}, f)
    print(f'  ✅ Scaler + meta → {pkl_path}')

    return meta

print('✅ Pipeline de entrenamiento definido')

In [ ]:
# ── Celda 7: Entrenar todos los tickers ──────────────────────────────────────
results = {}
failed  = []

for ticker in TICKERS:
    try:
        meta = train_ticker(ticker)
        results[ticker] = meta
    except Exception as e:
        print(f'  ❌ Error en {ticker}: {e}')
        failed.append(ticker)

print(f'\n{'='*50}')
print(f'✅ Entrenados: {len(results)} | ❌ Fallidos: {len(failed)}')
if failed:
    print(f'  Fallidos: {failed}')

# Summary
for t, m in results.items():
    print(f'  {t}: MAPE={m["mape"]}% | features={m["input_size"]}')

In [ ]:
# ── Celda 8: Verificar inferencia ONNX (test antes de subir) ─────────────────
import onnxruntime as ort

def test_onnx_inference(ticker: str):
    onnx_path = f'models/lstm_{ticker}.onnx'
    pkl_path  = f'models/lstm_{ticker}.pkl'

    with open(pkl_path, 'rb') as f:
        bundle = pickle.load(f)
    meta     = bundle['meta']
    scaler_y = bundle['scaler_y']

    sess  = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    X_win = np.array(meta['last_X_scaled'], dtype=np.float32).reshape(1, TIMESTEPS, -1)

    # Single inference
    pred_norm = sess.run(None, {'input': X_win})[0][0]
    pred_ret  = float(scaler_y.inverse_transform([[pred_norm]])[0][0])

    print(f'{ticker}: pred_ret_1d = {pred_ret*100:.3f}%  |  last_close = ${meta["last_close"]:.2f}')
    return True

print('Test de inferencia ONNX:')
for ticker in list(results.keys())[:3]:
    test_onnx_inference(ticker)

In [ ]:
# ── Celda 9: Descargar modelos ────────────────────────────────────────────────
import shutil
from google.colab import files

# Crear zip con todos los modelos
shutil.make_archive('stocklens_models', 'zip', 'models')
files.download('stocklens_models.zip')
print('✅ Descarga iniciada — descomprime y sube a stocklens-python/app/models/')